# 04 - Video Testing

This notebook:

- runs inference on videos
- generates a labeled video
- checks the average likelihood

## Cell 1 — Library Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd
import deeplabcut

ROOT_NOTEBOOK = Path.cwd().resolve().parent
if str(ROOT_NOTEBOOK) not in sys.path:
    sys.path.append(str(ROOT_NOTEBOOK))

from src.paths import CONFIG_PATH, VIDEO_SWEEP, VIDEO_RANDOM, VIDEO_RAMP, VIDEOS_DIR, show_paths

show_paths()
assert CONFIG_PATH.exists(), f"config.yaml not found: {CONFIG_PATH}"

Loading DLC 3.0.0rc13...
ROOT_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1
VIDEOS_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1\videos
RESULTS_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1\results
PROJECT_DIR: C:\Users\edoal\ProjetosPy\bab_dlc1\bab_bar_only-Edo-2026-03-08
CONFIG_PATH: C:\Users\edoal\ProjetosPy\bab_dlc1\bab_bar_only-Edo-2026-03-08\config.yaml


## Cell 2 — Select Video

In [ ]:
# Change only this variable
VIDEO_PATH = VIDEO_RAMP # VIDEO_SWEEP or VIDEO_RANDOM

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
print("Selected video:", VIDEO_PATH)

Selected video: C:\Users\edoal\ProjetosPy\bab_dlc1\videos\rampa_positiva.mp4


## Cell 3 — Analyze Video

In [ ]:
deeplabcut.analyze_videos(str(CONFIG_PATH), [str(VIDEO_PATH)], save_as_csv=True, snapshot_index=2)
deeplabcut.create_labeled_video(str(CONFIG_PATH), [str(VIDEO_PATH)], snapshot_index=2) # index 2 forces the use of snapshot 100.

Analyzing videos with C:\Users\edoal\ProjetosPy\bab_dlc1\bab_bar_only-Edo-2026-03-08\dlc-models-pytorch\iteration-0\bab_bar_onlyMar8-trainset80shuffle1\train\snapshot-100.pt
Using scorer: DLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100
Starting to analyze C:\Users\edoal\ProjetosPy\bab_dlc1\videos\rampa_positiva.mp4
Video metadata: 
  Overall # of frames:    2278
  Duration of video [s]:  75.93
  fps:                    30.0
  resolution:             w=1920, h=1080

Running pose prediction with batch size 4


100%|██████████████████████████████████████████████████████████████████████████████| 2278/2278 [14:10<00:00,  2.68it/s]


Saving results in C:\Users\edoal\ProjetosPy\bab_dlc1\videos\rampa_positivaDLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100.h5 and C:\Users\edoal\ProjetosPy\bab_dlc1\videos\rampa_positivaDLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100_full.pickle
The videos are analyzed. Now your research can truly start!
You can create labeled videos with 'create_labeled_video'.
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.

Starting to process video: C:\Users\edoal\ProjetosPy\bab_dlc1\videos\rampa_positiva.mp4
Loading C:\Users\edoal\ProjetosPy\bab_dlc1\videos\rampa_positiva.mp4 and data.
Duration of video [s]: 75.93, recorded with 30.0 fps!
Overall # of frames: 2278 with cropped frame dimensions: 1920 1080
Generating frames and creating video.


100%|██████████████████████████████████████████████████████████████████████████████| 2278/2278 [01:13<00:00, 30.99it/s]


[True]

## Cell 4 — Find the Generated H5 File

In [ ]:
stem = VIDEO_PATH.stem
h5_candidates = sorted(VIDEOS_DIR.glob(f"{stem}*.h5"))
assert h5_candidates, f"No .h5 file found for {stem}"

for i, p in enumerate(h5_candidates):
    print(i, p.name)

h5_path = h5_candidates[0]
print("Using:", h5_path)

0 random_steps_WDLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100.h5
Using: C:\Users\edoal\ProjetosPy\bab_dlc1\videos\random_steps_WDLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100.h5


### Célula 5 — likelihood

In [ ]:
df = pd.read_hdf(h5_path)

mean_likelihood = df.xs("likelihood", level="coords", axis=1).mean()
print("Average likelihood:")
print(mean_likelihood)

frac_baixa = (df.xs("likelihood", level="coords", axis=1) < 0.6).mean()
print("\nFraction below 0.6:")
print(frac_baixa)

Average likelihood:
scorer                                              bodyparts  
DLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100  beam_left      0.940726
                                                    beam_center    1.000000
                                                    beam_right     0.989052
dtype: float32

Fraction below 0.6:
scorer                                              bodyparts  
DLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100  beam_left      0.0
                                                    beam_center    0.0
                                                    beam_right     0.0
dtype: float64


## Cell 6 — Run for Other Videos

In [2]:
videos = [VIDEO_SWEEP, VIDEO_RANDOM, VIDEO_RAMP]

for video in videos:
    stem = video.stem
    h5_candidates = sorted(VIDEOS_DIR.glob(f"{stem}*.h5"))
    if not h5_candidates:
        print(f"No .h5 file for {video.name}")
        continue

    dfv = pd.read_hdf(h5_candidates[0])
    mean_likelihood = dfv.xs("likelihood", level="coords", axis=1).mean()

    print("\nVideo:", video.name)
    print(mean_likelihood)


Video: swept_sine_ready.mp4
scorer                                              bodyparts  
DLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100  beam_left      0.968387
                                                    beam_center    0.999937
                                                    beam_right     0.975720
dtype: float32

Video: random_steps_W.mp4
scorer                                              bodyparts  
DLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100  beam_left      0.940726
                                                    beam_center    1.000000
                                                    beam_right     0.989052
dtype: float32

Video: rampa_positiva.mp4
scorer                                              bodyparts  
DLC_Resnet50_bab_bar_onlyMar8shuffle1_snapshot_100  beam_left      0.934171
                                                    beam_center    1.000000
                                                    beam_right     0.972884
dtype: floa